# MedQA Correctness Cuboids

This notebook computes the correctness cuboids for a Q=300 MedQA test.The questions are multiple choice with 5 choices each.

In [1]:
import csv, collections, itertools, time
import ntqr

decisions_with_gt = []

decisions_file = "./MedQA4LLMsQ300.csv"
with open(decisions_file, mode='r', newline='') as file:
    reader = csv.reader(file)
    
    # Skip the header row
    header = next(reader)
    print(header)
    
    # Process the remaining data rows, we keep only
    # the responses and strip the ground truth.
    for row in reader:
        decisions_with_gt.append(tuple(d.lower() for d in row))

clean_dwgt = [row for row in decisions_with_gt if 'n' not in row]
print(len(clean_dwgt))

# What is the ground truth Q-simplex point?
print(collections.Counter([row[0] for row in clean_dwgt]))

observed_decisions = [row[1:] for row in clean_dwgt]
print(observed_decisions[50])

['gt', 'gemma_3n_it_agent_0', 'gpt-4o-mini_agent_0', 'llama-3.1-8b-chat_agent_0', 'mistral-7b_agent_0']
295
Counter({'a': 65, 'b': 63, 'c': 56, 'd': 56, 'e': 55})
('d', 'b', 'c', 'b')


## Computing the consistent set
The fundamental calculation in any logic of unsupervised evaluation is find the possible set of evaluations given how test takers disagree on a test. NTQR is now fast enough to make these computations feasible for research purposes.

The demonstration below highlights two aspects of NTQR's utility:
1. It is a semantically-free counting logic that applies to any multiple-choice exam as well, here results from giving 4 LLMs MedQA diagnosis questions.
2. Realistic number of questions, labels and classifiers can now be handled relatively quickly at any given answer-key simplex point.


In [2]:
import ntqr.evaluations

# The user defines how to denote the different labels
# and classifiers.
labels=('a','b','c','d','e')
classifiers=(1,2,3,4)

# Evaluation sets are 'parametrized' by their point in the
# answer-key Q-simplex. There are Q=295 questions in the
# cleaned up joint decisions. We pick a 'centroid' in that
# Q-simplex. As one would expect, this is the max in
# combinatorial options.
ql = (59,59,59,59,59)
print(sum(ql))
# We'll also have the ground truth Q-simplex point
gt_ql = (65,63,56,56,55)
print(sum(gt_ql))

# We collect the joint decision events using collections.Counter
counts = collections.Counter(observed_decisions)
cSet = ntqr.evaluations.ConsistentSet(labels, classifiers, counts)

295
295


In [3]:
# Let's time 100 points from the exact consistent set generator
import time
start = time.perf_counter()
res = list(itertools.islice(cSet.set_generator(ql),100))
end = time.perf_counter()

print(f"Actual calculation time: {end - start:.4f} seconds")

Actual calculation time: 0.5197 seconds


In [4]:
# We can validate these points using the axioms
# Each point is attached to the same Q-simplex point
# This verifies they sum to the given ql point.
collections.Counter([cSet.validate_point(point,ql) for point in res])

Counter({(True, 'Valid'): 100})

In [5]:
res[0]

(<Compressed Sparse Row sparse matrix of dtype 'int64'
 	with 31 stored elements and shape (1, 625)>,
 <Compressed Sparse Row sparse matrix of dtype 'int64'
 	with 26 stored elements and shape (1, 625)>,
 <Compressed Sparse Row sparse matrix of dtype 'int64'
 	with 27 stored elements and shape (1, 625)>,
 <Compressed Sparse Row sparse matrix of dtype 'int64'
 	with 27 stored elements and shape (1, 625)>,
 <Compressed Sparse Row sparse matrix of dtype 'int64'
 	with 31 stored elements and shape (1, 625)>)

## Joint evaluations are unique
Consistent evaluations are sparse in the possible set, but there are still many of them. Therefore, we can test if the `ConsistentSet.set_generator` code is correct in two ways:
1. For small Q values, we can compute the full list generated by the generator and check if all the members are singletons.
2. For large Q values, we can only check starting portions of the stream for uniqueness.
3. Testing by random sampling does not solve this because the code is different for random sampling than deterministic.

We test the random sampler here for minimum competency - the consistent set is so large that all values should be unique. This code is slower than the exact generator with the current version of NTQR.

In [6]:
import time

start = time.perf_counter()
res = list(itertools.islice(cSet.random_set_generator(ql), 10))
end = time.perf_counter()

print(f"Actual calculation time: {end - start:.4f} seconds")

Actual calculation time: 2.0923 seconds


In [7]:
# Let's test two points
cSet.are_points_equal(res[3],res[4])

False

## The correctness cuboid for this test
Now that the generators in `ConsistentSet` are faster, so are the exact and random generators for the correctness cuboids. Here we create a sample of 1000 at the our chosen Q-simplex point.

In [4]:
import time
start = time.perf_counter()
cc_points = list(itertools.islice( cSet.correct_cuboid_random_generator(gt_ql), 100))
end = time.perf_counter()
print(f"Actual calculation time: {end - start:.4f} seconds")
cc_points.sort(key=lambda item: sum([c/65 for c in item[0]]))
cc_points.reverse()
cc_points[:3]

KeyboardInterrupt: 

In [4]:
import cProfile
cProfile.run('list(itertools.islice( cSet.correct_cuboid_random_generator(gt_ql), 100))')

         16831049 function calls (16782932 primitive calls) in 16.575 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        4    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:1390(_handle_fromlist)
     2000    0.001    0.000    0.001    0.000 <frozen importlib._bootstrap>:645(parent)
      2/1    0.002    0.001    3.770    3.770 <string>:1(<module>)
     1000    0.001    0.000    0.001    0.000 _base.py:138(__init__)
     2000    0.001    0.000    0.002    0.000 _base.py:1512(_process_toarray_args)
     1500    0.001    0.000    0.010    0.000 _base.py:1525(_get_index_dtype)
     3000    0.001    0.000    0.004    0.000 _base.py:365(nnz)
     1000    0.000    0.000    0.000    0.000 _base.py:94(ndim)
     3500    0.001    0.000    0.001    0.000 _base.py:98(_shape_as_2d)
     2000    0.002    0.000    0.007    0.000 _compressed.py:1002(toarray)
      500    0.001    0.000    0.002    0.000 _compressed.py